# Federated Inference: Methods Comparison

## Load and pre-process data

In [ ]:
library(dplyr)
library(ggplot2)
library(tidyr)
library(confeR)

source("scripts/functions.R")

set.seed(123)

# Select data set

## Logistic regression
dataname <- "trauma"
dataname <- "trauma_shuffled"

# try this too?
# data(covid, package = "pda")
# data.frame(covid)

## Linear regression

dataname <- "Nurses"
#dataname <- "LOS"

#dataname <- "Nurses_shuffled"

#dataname <- "sim-linear-1-out"
#dataname <- "sim-linear-1"
#dataname <- "sim-nurses-1-out"
# dataname <- "sim-nurses-1"

# Name of the column designating the center id
center_name <- "hospital"

# Only implemented for linear regression for now
use_local_intercepts <- !startsWith(dataname, "trauma")
use_local_intercepts <- TRUE
use_local_variances <- FALSE

if (startsWith(dataname, "trauma")) {
        data(trauma, package = "BFI")
        data_raw <- trauma
} else if (startsWith(dataname, "Nurses")) {
        data(Nurses, package = "BFI")
        data_raw <- Nurses
} else if (startsWith(dataname, "LOS")) {
        data(LOS, package = "pda")
        data_raw <- LOS
        data_raw <- rename(data_raw, !!center_name := site)
        data_raw$hospital <- as.factor(sub("^site", "", data_raw$hospital))
} else {
        data_raw <- read.csv(paste0("data/raw/", dataname, ".csv"), row.names = 1)
}

# Shuffling center simulates homogeneous case
if (endsWith(dataname, "_shuffled")) {
        data_raw[[center_name]] <- sample(data_raw[[center_name]])
}

raw_dataname <- dataname
if (use_local_intercepts) {
    dataname <- paste0(dataname, "_", "local_int")
}
if (use_local_variances) {
    dataname <- paste0(dataname, "_", "local_var")
}

data_raw[[center_name]] <- factor(data_raw[[center_name]],
                        levels = sort(unique(data_raw[[center_name]]),
                        decreasing = FALSE))
dataname
head(data_raw)

In [ ]:
table(data_raw[[center_name]])
median(table(data_raw[[center_name]]))
sort(unique(data_raw[[center_name]]), decreasing = FALSE)

In [ ]:
scale_numeric_cols <- function(data) {
  data[] <- lapply(data, function(x) if (is.numeric(x)) scale(x)[, 1] else x)
  data
}

scale_all_cols <- function(data, center = TRUE, scale = TRUE) {
    require(dplyr)
    data |>
        mutate(across(where(is.numeric), ~ as.numeric(scale(.x, center = center, scale = scale))))
}

one_hot <- function(df, cols_to_encode) {
    dummies <- model.matrix(~ . - 1, data = df[cols_to_encode])
    cbind(df[setdiff(names(df), cols_to_encode)], dummies)
}

if (startsWith(dataname, "trauma")) {
    data_norm <- data_raw |>
        # don't need to normalize within group as combined mean and sd can be computed without combining data
        mutate(
            age = scale(age),
            ISS = scale(ISS),
            GCS = scale(GCS)
        )

    model <- mortality ~ sex + age + ISS + GCS
    family <- "binomial" #(link = "logit")

} else if (startsWith(dataname, "Nurses") || startsWith(dataname, "sim-nurses")) {
    data_norm <- data_raw |>
    mutate(
            age = scale(age),
            experience = scale(experien),
            stress = scale(stress),

            # keep numeric for now; fine if binary 0, 1
            #gender = factor(gender),
            #wardtype = factor(wardtype)
    )
    #data_norm <- scale_numeric_cols(data_raw) |> mutate(experience=experien)

    if (use_local_intercepts) {
        model_no_int <- as.formula(paste("stress ~ 0 + gender + age + experience +", center_name))
        model <- stress ~ gender + age + experience  # BFI paper leaves out wardtype
    } else {
        model <- stress ~ gender + age + experience + wardtype
    }
    family <- "gaussian"
} else if (startsWith(dataname, "LOS")) {
    data_norm <- data_raw |>
    mutate(
            lab = scale(lab),
            los = scale(los),
    )
    data_norm <- one_hot(data_norm, c("age", "sex")) |> rename("sex"="sexM")
    model_no_int <- as.formula(paste("los ~ 0 + sex + ageyoung + ageold + lab +", center_name))
    model <- los ~ sex + ageyoung + ageold + lab
    family <- "gaussian"

} else if (startsWith(dataname, "sim-linear")) {
    data_norm <- scale_numeric_cols(data_raw)
    predictors <- setdiff(names(data_norm), c("y", "hospital"))
    if (use_local_intercepts) {
        model_no_int <- as.formula(paste("y ~ 0 +", paste(predictors, collapse = " + "), "+ hospital"))
        model <- as.formula(paste("y ~", paste(predictors, collapse = " + ")))
    } else {
        model <- as.formula(paste("y ~", paste(predictors, collapse = " + ")))
        m_brms <- as.formula(paste("y ~ 0 + Intercept +", paste(predictors, collapse = " + ")))
    }
    family <- "gaussian"
    family_alt <- family
} else {
  stop()
}

covariates <- attr(terms(model), "term.labels")
#covariates_local <- covariates[covariates != center_name]
outcome <- all.vars(model)[1]

if (center_name %in% covariates) {
    data <- data_norm[c(outcome, covariates)]
} else {
    data <- data_norm[c(center_name, outcome, covariates)]
}

sites <- levels(data[[center_name]])

data_split <- data |>
  group_by(.data[[center_name]]) |>
  group_split()

n_sites <- length(data_split)
head(data_split[[1]])
n_data <- sum(unlist(lapply(data_split, nrow)))

if (use_local_intercepts) {
    p <- length(covariates) + n_sites
} else {
    p <- length(covariates)
}
p

# Using Bayesian Conjguate Analysis

In [ ]:
bstats <- bca_iterate_sites(outcome, covariates, model, family, data_split, use_local_intercepts, center_name)
params_oneshot <- bca_oneshot(bstats, n_sites, use_local_intercepts, use_local_variances, family,
                              epsilon = 1e-10, center_name)

df_bca <- tidy_results(params_oneshot, use_local_intercepts)
df_bca

In [ ]:
## Pseudo-data generation for "frequentist prior"

# cid <- 1
# #bl <- bayes_local_glm(bstats, center_identity=cid, family=family, alpha=0.05)

# bstats_cid <- bca_iterate_sites(outcome, covariates, model, family,
#                                 data_split[cid], use_local_intercepts, center_name)
# params_oneshot_cid <- bca_oneshot(bstats[cid], 1, use_local_intercepts, use_local_variances, family,
#                                  epsilon = 1e-10, center_name)

# generate_pseudo_data <- function(beta, lambda, a) {
#     require(expm)

#     x <- sqrtm(lambda)
#     y <- x %*% beta
#     n <- 2*a + nrow(lambda)
#     xx <- t(x) %*% x
#     xy <- t(x) %*% y
#     yy <- t(y) %*% y
#     list("n"=n, "xx"=xx, "xy"=xy, "yy"=yy)
# }

# b <- params_oneshot_cid$beta_l
# l <- params_oneshot_cid$lambda_l
# a <- params_oneshot_cid$a_l
# psd <- generate_pseudo_data(b, l, a)
# all.equal(psd$xx, unname(bstats[[cid]]$xx))
# all.equal(psd$yy, unname(bstats[[cid]]$yy))
# all.equal(psd$xy, unname(bstats[[cid]]$xy))
# all.equal(psd$n, unname(bstats[[cid]]$n))

# Using Multivariate Meta-Analysis

In [ ]:
# Fit local GLMs
local_glms <- fit_local_glms(data_split, model, outcome, covariates, center_name)
coef_list <- local_glms$coef_list
se_list <- local_glms$se_list
cov_list <- local_glms$cov_list

In [ ]:
# Meta-analyse GLM parameters
df_meta_mv_fe <- fit_mv_meta_fixed(coef_list, cov_list)
df_meta_mv_reml <- fit_mv_meta_random(coef_list, cov_list, method="REML")

# add simple mean of error variances
if (family == "gaussian") {
    sigma2 <- mean(local_glms$sigma2_list)

    use_boot_CI <- FALSE
    if (use_boot_CI) {
        boot_means <- replicate(10000, mean(sample(local_glms$sigma2_list, replace = TRUE)))
        ci <- quantile(boot_means, c(0.025, 0.975))
        lower <- ci[[1]]
        upper <- ci[[2]]
    } else {
        sigma2_sd <- sd(local_glms$sigma2_list)
        n <- length(local_glms$sigma2_list)
        error_margin <- qt(0.975, df = n - 1) * sigma2_sd / sqrt(n)
        lower <- sigma2 - error_margin
        upper <- sigma2 + error_margin
    }

    row <- list("sigma2", sigma2, lower, upper, "FE")
    df_sigma2 <- as.data.frame(row, stringsAsFactors = FALSE)
    colnames(df_sigma2) <- colnames(df_meta_mv_fe)
    df_meta_mv_fe <- rbind(df_meta_mv_fe, df_sigma2)

    row <- list("sigma2", sigma2, lower, upper, "REML")
    df_sigma2 <- as.data.frame(row, stringsAsFactors = FALSE)
    colnames(df_sigma2) <- colnames(df_meta_mv_fe)
    df_meta_mv_reml <- rbind(df_meta_mv_reml, df_sigma2)
}
df_meta_mv_fe
df_meta_mv_reml

# Using BFI

In [ ]:
res_bfi_sub <- bfi_sub(data_split, family, outcome, covariates, center_name)
df_bfi <- fit_bfi(data_split, family, res_bfi_sub, use_local_intercepts, use_local_variances)

# Using PDA

In [ ]:
fit.pda <- fit_pda(outcome, covariates, use_local_intercepts, data_split, sites, dataname, family, dir="data/pda")
df_pda <- tidy_pda(fit.pda, family, use_local_intercepts, covariates, n_sites, n_data, alpha=0.05)

# Using combined data (ground truth)

In [ ]:
df_combined <- fit_combined_glm(data, family, model, model_no_int, use_local_intercepts, center_name)

# Compare methods

In [ ]:
library(ggplot2)
library(ggpubr)
library(stringr)

method_order <- c("Combined", "BCA", "PDA", "BFI", "FE", "REML")
df_merged <- rbind(df_combined, df_bca, df_pda, df_bfi, df_meta_mv_fe, df_meta_mv_reml)  # must match above

if (use_local_intercepts) {
  df_merged <- df_merged |>
    filter(
      Covariate != "(Intercept)",
      !str_starts(Covariate, "Intercept_")
    )
}

df_merged$Method <- factor(df_merged$Method, levels = method_order)
write.csv(df_merged, paste0("data/summarized/params.", dataname, ".csv"))

huemap <- setNames(get_palette(palette = "npg", length(levels(df_merged$Method))), levels(df_merged$Method))

In [ ]:
plot_params <- function(dataname, df_merged) {
  p <- ggplot(df_merged, aes(x = Method, y = Estimate, color = Method)) +
  geom_point(position = position_dodge(width = 0.5)) +
  geom_errorbar(aes(ymin = lower, ymax = upper), 
                width = 0.2,
                position = position_dodge(width = 0.5)) +
  scale_color_manual(values = huemap) +
  facet_wrap(~ Covariate, scales = "free_y") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(y = "Estimate ± 95% CI", x = NULL)

  ggsave(paste0("figures_tmp/params.", dataname, ".png"), bg = "white")
  p
}

plot_params(dataname, df_merged)

# if (startsWith(dataname, "sim-")) {
#     simtruth <- read.csv(paste0("data/raw/", raw_dataname, ".truth.csv"))
#     simtruth
# }

In [ ]:
if (family == "gaussian" && use_local_intercepts) {

    shapiro.test(fit.pda$uhat)
    par(bg = "white")
    qqnorm(fit.pda$uhat) + theme_minimal()
    qqline(fit.pda$uhat, col = "red")

    x <- df_combined[startsWith(df_combined$Covariate, "Intercept_"), "Estimate"]
    shapiro.test(x)
    par(bg = "white")
    qqnorm(x) + theme_minimal()
    qqline(x, col = "red")

    plot(x, df_pda$Estimate[1:n_sites])
}

# Reverse-Bayes

In [ ]:
stopifnot(family=="gaussian")  # not yet implemented

In [ ]:
box_results <- box_check_all_sites(data_split, params_oneshot, bstats, family, covariates,
                                n_sites, use_local_intercepts, center_name, remove_intercept = use_local_intercepts)
pboxes <- box_results$pboxes
write.csv(pboxes, paste0("data/summarized/pboxes.", dataname, ".csv"))

In [ ]:
cid <- 9
-log2(pboxes)
-log2(pboxes)[cid]
local_glms$sigma2_list
as.vector(t(lapply(data_split, nrow)))
mean(local_glms$sigma2_list)

In [ ]:
ggplot(data=NULL, aes(y = -log2(pboxes), x = local_glms$sigma2_list)) +
  geom_point(color = "steelblue") +
  theme_minimal()

## Box prior predictive check

In [ ]:
gghistogram(as.numeric(pboxes), fill = "lightgray", bins=10)

In [ ]:
df <- data.frame(Center = seq_along(pboxes), neg_log_p = -log10(pboxes))

ggplot(df, aes(x = Center, y = neg_log_p)) +
  geom_point(color = "steelblue") +
  theme_minimal() +
  labs(
    x = "Center",
    y = expression(-log[10](p)),
    title = "Manhattan plot for pbox"
  ) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "red")


## Forest plot

In [ ]:
df_forest <- prepare_forest_plot(df_bca, bstats, alpha=0.05)
df_forest <- df_forest |> filter(df_forest$Covariate != "sigma2")
forest_plot(df_forest, pboxes, outfile = paste0("figures_tmp/forest.", dataname, ".pdf"),
            inline_plot = TRUE, use_log_scale = TRUE, order_box = TRUE)

In [ ]:
# sigma2: Forest bca_oneshot with 1 site equals local glm
aa <- unname(unlist(as.vector(df_forest |> filter(Covariate == "sigma2" & site != "Federated") |> select(Estimate))))
bb <- local_glms$sigma2_list
all.equal(aa, bb)

## Leverage

In [ ]:
df_long <- reduced_params |>
    pivot_longer(cols = -center_name, names_to = "variable", values_to = "value")

df_long <- df_long[!startsWith(df_long$variable, "Intercept_"), ]

p <- ggboxplot(
  df_long,
  y = "value",
  color = "variable",
  add = "jitter",
  facet.by = "variable",
  palette = "jco"
)
p + facet_wrap(~variable, scales = "free_y")

## Cross-validation of prognostic power

In [ ]:
library(yardstick)
library(dplyr)

cat("Number of local centers:", length(data_split), "\n\n")

make_pred_logistic <- function(data, beta, formula) {
    x <- model.matrix(formula, data)
    logits <- x %*% beta
    probs <- 1 / (1 + exp(-logits))
    pred <- ifelse(probs >= 0.5, 1, 0)
    list("pred" = pred, "probs" = as.vector(probs))
}

eval_logistic <- function(truth, pred, probs = NULL)  {
    df_eval <- tibble(
        truth = as.factor(truth),
        pred = as.factor(pred)
    )
    df_eval <- df_eval |>
    mutate(across(c(truth, pred), ~ factor(.x, levels = c(0, 1))))
    metrics <- yardstick::metric_set(accuracy, precision, recall, f_meas)(df_eval, truth = truth, estimate = pred)

    if (!is.null(probs)) {
        df_eval$probs <- probs
        roc_auc_ <- yardstick::roc_auc(df_eval, truth, probs, event_level = "second")
        metrics <- rbind(metrics, roc_auc_)
    }
    metrics
}

eval_linear <- function(truth, data, beta, formula, use_local_intercepts) {

    if (use_local_intercepts) {
        formula <- update(formula,    ~ . + hospital)
        x <- model.matrix(formula, data)
        colnames(x) <- sub("^hospital(\\d+)$", "Intercept_\\1", colnames(x))
        colnames(x) <- sub("\\(Intercept\\)", "Intercept_1", colnames(x))
    } else {
        x <- model.matrix(formula, data)
    }
    eta <- as.vector(x %*% beta)

    residuals <- truth - eta
    mse <- mean(residuals^2)
    rmse <- sqrt(mse)
    mae <- mean(abs(residuals))

    ss_total <- sum((truth - mean(truth))^2)
    ss_res <- sum((residuals)^2)
    r_squared <- 1 - ss_res / ss_total

    metrics <- list(
        RMSE = rmse,
        MAE = mae,
        R2 = r_squared,
        MSE = mse
    )
    list(metrics=metrics, x=x, eta=eta)
}

if (family == "binomial") {
    l <- 1
    pred_probs <- make_pred_logistic(data_split[[l]], beta[[l]], model)
    pred <- pred_probs$pred
    probs <- pred_probs$probs
    truth <- data_split[[l]][[outcome]]
    table(Predicted = pred, Actual = truth)

    eval_logistic(truth, pred, probs)

} else if (family == "gaussian") {
    res <- eval_linear(data[[outcome]], data, params_oneshot$beta, model, use_local_intercepts)
    metrics <- res$metrics
    x <- res$x
    eta <- res$eta

    p <- ggplot(data, aes(x = .data[[outcome]], y = eta)) +
    geom_point() +
    # Identity line (y = x)
    geom_abline(slope = 1, intercept = 0, color = "grey", linetype = "dashed") +
    # Linear fit line (data-based)
    geom_smooth(method = "lm", se = FALSE, color = "red") +
    labs(
        x = outcome,
        y = "Predicted",
        title = paste("Predicted vs", outcome)
    ) +
    theme_minimal()
    p

} else {
    stop(paste0("Invalid family:", family))
}


In [ ]:
metrics <- vector("list", length(data_split))

if (family == "binomial") {

    for (l in seq_along(data_split)) {
        beta_minus_l <- as.numeric(reduced_params[reduced_params[center_name]==l, rownames(params_oneshot$beta)])
        pred_probs <- make_pred_logistic(data_split[[l]], beta_minus_l, model)
        pred <- pred_probs$pred
        probs <- pred_probs$probs
        truth <- data_split[[l]]$mortality
        metrics[[l]] <- eval_logistic(truth, pred, probs)
    }

    metrics <- bind_rows(metrics, .id = "local_center")

} else if (family == "gaussian") {

    bayes_post_params <- params_oneshot$post_params

    for (l in seq_along(data_split)) {

        beta_minus_l <- as.numeric(reduced_params[reduced_params[center_name]==l, rownames(params_oneshot$beta)])
        res <- eval_linear(data_split[[l]][[outcome]], data_split[[l]], beta_minus_l, model, use_local_intercepts)
        metrics[[l]] <- res$metrics
    }

    metrics <- bind_rows(metrics, .id = "local_center")
    metrics <- metrics |>
        pivot_longer(cols = c(RMSE, MAE, R2, MSE),
               names_to = "Metric",
               values_to = "Value")

} else {
    stop(paste0("Invalid family:", family))
}

write.csv(metrics, paste0("data/summarized/cross-metrics.", dataname, ".csv"))

In [ ]:
library("ggpubr")

if (family == "binomial") {

    ggboxplot(
        metrics,
        x = ".metric",
        y = ".estimate",
        color = ".metric",
        add = "jitter",
        palette = "jco"
    ) +
    theme_minimal()

} else if (family == "gaussian") {
    ggboxplot(metrics,
        x = "Metric",
        y = "Value",
        color = "Metric",
        add = "jitter",
        palette = "jco") +
    theme_minimal() +
    labs(title = paste("Linear regression metrics for", length(data_split), "local centers"),
        x = "Metric",
        y = "Metric Value")
}

In [ ]:
stop()

# Normal-(inverse)-gamma

In [ ]:
covariates_local <- covariates[covariates != center_name]
prior_params <- bs$get_linreg_prior(covariates_local, use_local_intercepts, n_sites, epsilon=0)
bp <- bs$bayes_lin_reg_post_params(bstats, prior_params)
alpha <- bp$a_l
beta <- bp$b_l

In [ ]:
normal_gamma_plot <- function(bp, alpha, beta, covariates, covariate, use_local_intercepts, n_sites) {
    par(bg = "white")
    bp$mu_l[covariate, ]
    p <- length(covariates) + (if (use_local_intercepts) n_sites else 0)

    # Parameters for the Normal-Gamma
    mu0 <- bp$mu_l[covariate, ]
    lambda <- bp$lambda_l[covariate, covariate]
    alpha <- bp$a_l
    beta <- bp$b_l

    tau_l <- (alpha) / beta

    # Determine standard deviation of marginal mu (Student-t)
    df <- 2 * alpha
    scale2 <- beta / (lambda * alpha)
    sd_mu <- sqrt(as.vector(scale2 * df / (df - 2)))  # Only valid if df > 2

    # x-axis (mu): center around mu0, span ±4 SDs
    x_lim <- mu0 + c(-4, 4) * sd_mu

    # y-axis (tau): use quantiles of Gamma distribution
    y_lim <- qgamma(c(0.001, 0.999), shape = alpha, rate = beta)

    # Create grid of mu and tau values
    mu_vals <- seq(x_lim[[1]], x_lim[[2]], length.out = 200)
    tau_vals <- seq(y_lim[[1]], y_lim[[2]], length.out = 200)
    grid <- expand.grid(mu = mu_vals, tau = tau_vals)

    # Compute joint density of Normal-Gamma
    dnorm_gamma <- function(mu, tau, mu0, lambda, alpha, beta) {
        dnorm(mu, mean = mu0, sd = sqrt(1 / (lambda * tau))) *
        dgamma(tau, shape = alpha, rate = beta)
    }

    grid$z <- with(grid, dnorm_gamma(mu, tau, mu0, lambda, alpha, beta))
    z_matrix <- matrix(grid$z, nrow = length(mu_vals), ncol = length(tau_vals))

    contour(mu_vals, tau_vals, z_matrix,
            xlab = expression(mu), ylab = expression(tau),
            main = "Normal-Gamma Joint Density Contour")

    abline(v = mu0, col = "blue", lty = 2, lwd = 2)
    abline(h = tau_l, col = "blue", lty = 2, lwd = 2)


    ##### Plot marginals ####

    ### Marginal for mu: Student-t ###
    df <- 2 * alpha
    scale2 <- beta / (lambda * alpha)
    s <- sqrt(as.vector(scale2))
    curve(dt((x - mu0) / s, df = df) / s,
        from = x_lim[[1]], to = x_lim[[2]], col = "blue", lwd = 2,
        main = "Marginal of mu (Student-t)", ylab = "Density", xlab = expression(mu))

    abline(v = mu0, col = "blue", lty = 2, lwd = 2)

    ### Marginal for tau: Gamma ###
    # Plot the Gamma density
    curve(dgamma(x, shape = alpha, rate = beta),
        from = y_lim[[1]], to = y_lim[[2]], col = "red", lwd = 2,
        main = "Marginal of tau (Gamma)", ylab = "Density", xlab = expression(tau))

    abline(v = tau_l, col = "blue", lty = 2, lwd = 2)
    legend("topleft",
        legend = c("tau_l"),
        col = c("blue"),
        lty = c(2, 3),
        lwd = 2,
        bty = "n")
}

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5)
#par(mfrow = c(2, 3), bg = "white")
par(mfrow = c(1, 3), bg = "white")

normal_gamma_plot(bp, alpha, beta, covariates, "gender", use_local_intercepts, n_sites)
normal_gamma_plot(bp, alpha, beta, covariates, "age", use_local_intercepts, n_sites)
normal_gamma_plot(bp, alpha, beta, covariates, "experience", use_local_intercepts, n_sites)

In [ ]:
normal_inv_gamma_plot <- function(bp, alpha, beta, covariates, covariate, use_local_intercepts, n_sites) {

    mu0 <- bp$mu_l[covariate, ]
    lambda <- bp$lambda_l[covariate, covariate]
    alpha <- as.vector(bp$a_l)
    beta <- as.vector(bp$b_l)

    p <- length(covariates) + (if (use_local_intercepts) n_sites else 0)
    tau_l <- (alpha) / beta
    sigma2_l <- 1/tau_l

    # Determine standard deviation of marginal mu (Student-t)
    df <- 2 * alpha
    scale2 <- beta / (lambda * alpha)
    sd_mu <- as.vector(sqrt(scale2 * df / (df - 2)))  # Only valid if df > 2

    # x-axis (mu): center around mu0, span ±4 SDs
    x_lim <- mu0 + c(-4, 4) * sd_mu

    # y-axis (tau): use quantiles of Gamma distribution
    qinvgamma <- function(p, shape, rate) {
        1 / qgamma(1 - p, shape = shape, rate = rate)
    }
    y_lim <- qinvgamma(c(0.001, 0.999), shape = alpha, rate = beta)

    # Create grid of mu and tau values
    mu_vals <- seq(x_lim[[1]], x_lim[[2]], length.out = 200)
    tau_vals <- seq(y_lim[[1]], y_lim[[2]], length.out = 200)
    grid <- expand.grid(mu = mu_vals, tau = tau_vals)

    dinv_gamma <- function(sigma2, alpha, beta) {
        log_density <- alpha * log(beta) - lgamma(alpha) - (alpha + 1) * log(sigma2) - beta / sigma2
        exp(log_density)
    }
    dnorm_inv_gamma <- function(mu, sigma2, mu0, lambda, alpha, beta) {
        s <- sqrt(sigma2 / lambda)
        dnorm(mu, mean = mu0, sd = s) * dinv_gamma(sigma2, alpha, beta)
    }


    grid$z <- with(grid, dnorm_inv_gamma(mu, tau, mu0, lambda, alpha, beta))
    z_matrix <- matrix(grid$z, nrow = length(mu_vals), ncol = length(tau_vals))

    contour(mu_vals, tau_vals, z_matrix,
            xlab = expression(mu), ylab = "sigma",
            main = "Normal-Inverse-Gamma Joint Density Contour")

    abline(v = mu0, col = "blue", lty = 2, lwd = 2)
    abline(h = sigma2_l, col = "blue", lty = 2, lwd = 2)


    ##### Plot marginals ####

    ### Marginal for mu: Student-t ###
    df <- 2 * alpha
    scale2 <- beta / (lambda * alpha)
    curve(dt((x - mu0) / sqrt(scale2), df = df) / sqrt(scale2),
        from = x_lim[[1]], to = x_lim[[2]], col = "blue", lwd = 2,
        main = "Marginal of mu (Student-t)", ylab = "Density", xlab = expression(mu))

    abline(v = mu0, col = "blue", lty = 2, lwd = 2)

    ### Marginal for sigma2: Inverse-Gamma ###
    curve(dinv_gamma(x, alpha, beta),
        from = y_lim[[1]], to = y_lim[[2]], col = "red", lwd = 2,
        main = "Marginal of sigma2 (Inverse-Gamma)", ylab = "Density", xlab = "sigma2")

    abline(v = sigma2_l, col = "blue", lty = 2, lwd = 2)

    legend("topleft",
        legend = c("sigma2_l"),
        col = c("blue"),
        lty = c(2, 3),
        lwd = 2,
        bty = "n")
}

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5)
par(mfrow = c(1, 3), bg = "white")

normal_inv_gamma_plot(bp, alpha, beta, covariates, "gender", use_local_intercepts, n_sites)
normal_inv_gamma_plot(bp, alpha, beta, covariates, "age", use_local_intercepts, n_sites)
normal_inv_gamma_plot(bp, alpha, beta, covariates, "experience", use_local_intercepts, n_sites)

In [ ]:
library(mvtnorm)
options(repr.plot.width = 10, repr.plot.height = 5)
par(mfrow = c(1, 2), bg = "white")

covariate1 <- "gender"
covariate2 <- "age"

alpha <- bp$a_l
beta <- bp$b_l
tau_l <- (alpha - p/2) / beta
tau_mode_uv_joint <- (alpha - 1/2) / beta

plot_mvn <- function(bp, covariate1, covariate2, tau) {

    lambda_12 <- bp$lambda_l[c(covariate1, covariate2), c(covariate1, covariate2)]
    mu1 <- bp$mu_l[covariate1, ]
    mu2 <- bp$mu_l[covariate2, ]

    mu <- c(mu1, mu2)
    Sigma <- solve(as.numeric(tau) * lambda_12)
    x_lim <- mu1 + c(-4, 4) * sqrt(Sigma[1, 1])
    y_lim <- mu2 + c(-4, 4) * sqrt(Sigma[2, 2])

    x <- seq(x_lim[[1]], x_lim[[2]], length.out = 100)
    y <- seq(y_lim[[1]], y_lim[[2]], length.out = 100)
    grid <- expand.grid(x = x, y = y)
    z <- matrix(dmvnorm(grid, mean = mu, sigma = Sigma), nrow = 100)

    contour(x, y, z, xlab = covariate1, ylab = covariate2)
}


plot_mvn(bp, covariate1, covariate2, tau_l)
plot_mvn(bp, covariate1, covariate2, tau_mode_uv_joint)

# TO DO:

- Support non-numeric site factors

In [ ]:
set.seed(123)

n <- 200
x <- runif(n, 0, 3)
beta0 <- 0.5
beta1 <- 0.8
mu <- exp(beta0 + beta1 * x)  # mean on natural scale

## Poisson data
y_pois <- rpois(n, lambda = mu)

## Negative Binomial data
# θ (size) controls overdispersion — smaller = more overdispersion
theta <- 2
y_nb <- rnbinom(n, mu = mu, size = theta)

dat <- data.frame(x, y_pois, y_nb)
head(dat)


In [ ]:
library(ggplot2)

dat$fit_pois <- fitted(m_pois)

ggplot(dat, aes(x, y_pois)) +
  geom_point(alpha = 0.6) +
  geom_line(aes(y = fit_pois), colour = "blue", size = 1) +
  labs(y = "Counts", title = "Poisson (blue) vs NegBin (red dashed)") +
  theme_minimal()


In [ ]:
m_pois <- glm(y_pois ~ x, family = poisson, data = dat)
summary(m_pois)
dispersion <- sum(residuals(m_pois, type = "pearson")^2) / m_pois$df.residual
dispersion
sigma(m_pois)^2

In [ ]:
shape <- 0
rate  <- 0

x <- seq(0, 10, length.out = 1000)
y <- dgamma(x, shape = shape, rate = rate)

plot(x, y, type = "l", lwd = 2, col = "blue",
     main = sprintf("Gamma Distribution (shape = %.1f, rate = %.1f)", shape, rate),
     xlab = "x", ylab = "Density")

abline(v = shape / rate, col = "red", lty = 2)  # mean
text(shape / rate, max(y) * 0.8, labels = "mean", pos = 4, col = "red")

 # SessionInfo

In [ ]:
sink("session_info.txt")
print(sessionInfo())
sessionInfo()

- berger pvalues for composite etc pbox$
- simulate pbox